# RoboMME-RI: perturbations and RI-Mem

This notebook presents selected clean-success / perturbed-failure pairs for FS+Mod, TD+Mod and SimpleMemVLA, with a multi-baseline evaluation entry point.

The protocol follows the main paper; RI-Mem follows supplement Sections 1 and 2.3. Videos are historical tr-v1/ev-2 examples. Their timing, delivery and budgets differ from the paper protocol; they provide qualitative observations rather than new-protocol or RI-Mem results.

In [ ]:
from pathlib import Path
import sys, json, hashlib
from IPython.display import display, Video, Image
cwd = Path.cwd().resolve()
PACKAGE = next(p for p in [cwd, *cwd.parents, cwd/'robustness_test/submitcodes'] if (p/'robomme_ri').is_dir())
sys.path.insert(0, str(PACKAGE))
NB = PACKAGE/'notebooks'
cases = json.loads((NB/'cases.json').read_text())
assert all(c['clean_status']=='success' and c['perturbed_status']!='success' for c in cases)
for case in cases:
    assert hashlib.sha256((NB/case['video']).read_bytes()).hexdigest() == case['sha256']
print(f'{len(cases)} verified pairs across {len(set(c["method"] for c in cases))} methods')

## Ten conditions

Clean and perturbed runs share the official task and environment seed. Each episode receives one condition. Bright/dark illumination and insertion/removal are subtypes. Events advance on every environment control step, including internal action-chunk and waypoint-planner steps.

In [ ]:
import pandas as pd
from dataclasses import asdict
from robomme_ri import CONDITIONS
display(pd.DataFrame([{'condition':k, **asdict(v)} for k,v in CONDITIONS.items()]))

## Matched qualitative pairs

Left: successful clean rollout. Right: failed or timed-out perturbed rollout. Instructions and provenance are preserved. Keyframes locate the behavior; interpreting failure mechanisms requires the full trajectory. A single paired outcome does not establish memory-specific causality.

### 1. perceptual-framesamp-modul / ButtonUnmaskSwap / episode 2

**Instruction:** first press both buttons on the table, then pick up the container hiding the blue cube

Event: `mid_arrive_early`; object: `button_left`; role: `answer_only`; actuation: `hand`.
Clean: success (348 steps). Perturbed: timeout (1485 steps).

[Paired video](assets/01_perceptual-framesamp-modul_ButtonUnmaskSwap_ep2.mp4)

![Keyframes](assets/01_perceptual-framesamp-modul_ButtonUnmaskSwap_ep2.jpg)


In [ ]:
case = cases[0]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

### 2. perceptual-tokendrop-modul / ButtonUnmaskSwap / episode 3

**Instruction:** first press both buttons on the table, then pick up the container hiding the green cube, finally pick up another container hiding the blue cube

Event: `on_arrive_sync`; object: `button_left`; role: `answer_only`; actuation: `slide`.
Clean: success (759 steps). Perturbed: fail (326 steps).

[Paired video](assets/02_perceptual-tokendrop-modul_ButtonUnmaskSwap_ep3.mp4)

![Keyframes](assets/02_perceptual-tokendrop-modul_ButtonUnmaskSwap_ep3.jpg)


In [ ]:
case = cases[1]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

### 3. perceptual-framesamp-modul / VideoUnmask / episode 0

**Instruction:** watch the video carefully, then pick up the container hiding the green cube

Event: `mid_takeback_early`; object: `bin_0`; role: `answer`; actuation: `slide`.
Clean: success (97 steps). Perturbed: fail (104 steps).

[Paired video](assets/03_perceptual-framesamp-modul_VideoUnmask_ep0.mp4)

![Keyframes](assets/03_perceptual-framesamp-modul_VideoUnmask_ep0.jpg)


In [ ]:
case = cases[2]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

### 4. simplememvla-robomme / BinFill / episode 10

**Instruction:** put one red cube and three green cubes into the bin, then press the button to stop

Event: `mid_displace`; object: `board_with_hole`; role: `answer_only`; actuation: `hand`.
Clean: success (1026 steps). Perturbed: fail (862 steps).

[Paired video](assets/04_simplememvla-robomme_BinFill_ep10.mp4)

![Keyframes](assets/04_simplememvla-robomme_BinFill_ep10.jpg)


In [ ]:
case = cases[3]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

## Unified RI-Mem

SCQ maintains the active subgoal. RGB regions form a causal evidence index. AMR scores relevance, explicit irrelevance, observability and temporal validity on keys already retained by the native policy. FS+Mod and TD+Mod receive bounded logit bias and output gating. SimpleMemVLA adds bias and log(gate) to historical video keys in full-attention layers 27 and 31.

The query, evidence features and 35-parameter head are shared; each backbone requires its own calibrated head. The numerical example illustrates the readout equations rather than an enhanced rollout.

In [ ]:
import numpy as np
from robomme_ri.rimem.head import initial_parameters, numpy_readout
W, beta = initial_parameters()
facts = np.array([[.9,0,.95,-1], [.9,0,0,1], [0,.95,0,0]], dtype=np.float32)
bias, gate = numpy_readout([1,0,0,0,1,1], facts)
display(pd.DataFrame({'evidence':['outdated spatial state','retained identity','explicit distractor'], 'bias':bias}))
print('Parameters:', W.size+beta.size, 'gate:', float(gate))

## Multi-baseline closed-loop evaluation

The configuration includes the 26 Table 2 baselines and three RI-Mem variants over sixteen tasks and fifty episodes, each under clean and ten perturbations. Configure native runtimes, checkpoints and service endpoints, and calibrate the RI-Mem heads before evaluation.

Episode-level admissibility and technical errors are reported separately. StopCube task-object perturbations remain excluded by the current conservative selector. Planning the full grid does not establish valid coverage or pretrained performance.

In [ ]:
from robomme_ri.cli import load_configuration, grid
from robomme_ri.baselines import METHODS
cfg = load_configuration(PACKAGE/'configs/robomme.json')
display(pd.DataFrame([{'method':m,'label':METHODS[m].label,'category':METHODS[m].category,'ri_mem':METHODS[m].rimem} for m in cfg['methods']]))
planned = sum(1 for _ in grid(cfg))
print('Planned closed-loop episodes:', planned)
RUN_EVALUATION = False
if RUN_EVALUATION:
    import subprocess
    subprocess.run([sys.executable,str(PACKAGE/'scripts/run_experiment.py'),str(PACKAGE/'configs/robomme.json'),
                    '--output',str(PACKAGE/'outputs/experiment'),'--gpus','0'],check=True)

In [ ]:
from robomme_ri.evaluate import summarize
paths = sorted((PACKAGE/'outputs/experiment').glob('**/rollouts.jsonl'))
if paths:
    rows = [json.loads(line) for p in paths for line in p.read_text().splitlines() if line.strip()]
    display(pd.DataFrame(summarize(rows)))
else:
    print('Run the evaluator to populate outputs/experiment.')